In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Get_luzard_chandler_hbonds)=
# Get Luzar-Chandler hydrogen bonds

*Detecting hydrogen bonds based on combined distance and angle criteria (Luzar & Chandler).*

The function {func}`molsysmt.interactions.hbonds.get_luzard_chandler_hbonds` evaluates hydrogen bonds using the joint donor–acceptor distance and H–D–A angle with the donor as vertex thresholds established by Luzar & Chandler.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.interactions.hbonds.get_luzard_chandler_hbonds`.
:::


## Basic usage

Let's detect hydrogen bonds in a protein structure using the Luzar-Chandler distance and angle criteria:


In [2]:
import molsysmt as msm
import pyunitwizard as puw


In [3]:
molsys = msm.convert(msm.systems['chicken villin HP35']['chicken_villin_HP35.h5msm'])


:::{note} Demo Systems Catalog
:class: dropdown

This tutorial uses demonstration datasets provided by MolSysMT. To explore the full catalog of bundled systems, forms, and file paths, visit the {ref}`Demo Systems <user-foundations-entrance-demo-systems>` guide.
:::


We evaluate hydrogen bonds with a distance cutoff of 3.5 Å and an angle tolerance of 30°:


In [4]:
hbonds, distances, angles = msm.interactions.hbonds.get_luzard_chandler_hbonds(molsys,
                                                                  distance_threshold='3.5 angstroms',
                                                                  angle_threshold='30.0 degrees')
print('Total Luzar-Chandler hydrogen bonds detected:', len(hbonds[0]))
print('First 5 H-bond triplets [donor_heavy, bonded_H, acceptor]:\n', hbonds[0][:5])
print('First 5 distances:\n', distances[0][:5])
print('First 5 angles:\n', angles[0][:5])


Total Luzar-Chandler hydrogen bonds detected: 36
First 5 H-bond triplets [donor_heavy, bonded_H, acceptor]:
 [[ 42  43  91]
 [ 51  52  90]
 [ 80  81  90]
 [ 92  93  47]
 [112 113  58]]
First 5 distances:
 [0.29376070029188095 0.25604054035224855 0.27401248994459554 0.2834160516724867 0.2878586446826376] nanometer
First 5 angles:
 [0.35409470074619875 0.10517367420842096 0.5053429033349056 0.30326356925490494 0.20589640163446343] radian


## Hydrogen bonds between selections

Hydrogen bonds can also be evaluated across two distinct subsets or candidate selections using `selection` and `selection_2`. Here we demonstrate this general feature with an example of interfacial hydrogen bonds formed between the two binding partners in the Barnase–Barstar complex:


In [5]:
complex_sys = msm.convert(msm.systems['Barnase-Barstar']['barnase_barstar.h5msm'])
inter_hbonds, inter_dist, inter_ang = msm.interactions.hbonds.get_luzard_chandler_hbonds(complex_sys,
                                                                            selection='molecule_name=="BARNASE"',
                                                                            selection_2='molecule_name=="BARSTAR"')
print('Interfacial hydrogen bonds detected:', len(inter_hbonds[0]))
print('Interface H-bond triplets:\n', inter_hbonds[0][:5])


Interfacial hydrogen bonds detected: 15
Interface H-bond triplets:
 [[ 431  432 2400]
 [ 431  433 2361]
 [ 858  859 2296]
 [ 876  878 2969]
 [ 879  880 2969]]


## Sparse results and evaluated coverage

Choose `output_type="molsysmt.Interactions"` for an independent analysis that you can query by atom and structure indices and attach explicitly under a name. It records D-A distance in nm, H-D-A angle in rad, the actual eligible atom scope, evaluated-empty structures, method parameters, evidence, and calculation-time software versions. With PBC, D-H and D-A images independently use the donor as reference and reproduce the angle calculation.

Automatic roles in one selection, two disjoint participant universes, or identical roles are supported. Supplied roles, a second structure axis, and partially overlapping universes are currently unsupported by the optional analysis. Calculation remains eager. See the {ref}`interaction result guide <user-tools-interactions-result>` for query semantics and persistence.


In [6]:
analysis = msm.interactions.hbonds.get_luzard_chandler_hbonds(
    molsys, output_type="molsysmt.Interactions"
)
molsys.interactions = {**molsys.interactions, "luzard_chandler": analysis}
visible = analysis.query(structure_indices=[0]).to_dict()
print("Sparse observations:", analysis.n_interactions)
print("Measurement units:", visible["measure_units"])
print("Producer version captured:", analysis.software["molsysmt"] == msm.__version__)


Sparse observations: 36
Measurement units: {'distance': 'nm', 'angle': 'rad'}
Producer version captured: True


For the default tuple output, equal counts across structures retain rectangular arrays. Varying counts return aligned lists of triples, distance quantities in nm, and angle quantities in radians. Each empty evaluated structure has shapes `(0, 3)`, `(0,)`, and `(0,)`; frames are not padded to a maximum count. The sparse result deduplicates repeated requested structures and distinguishes evaluated-empty frames from unevaluated ones.


:::{seealso} Related Tools & References
:class: dropdown

- {ref}`Get Buch hydrogen bonds <Tutorial_Get_buch_hbonds>`: Detect hydrogen bonds via Buch criteria with {func}`molsysmt.interactions.hbonds.get_buch_hbonds`.
- {ref}`Get acceptor atoms <Tutorial_Get_acceptor_atoms>`: Identify acceptor atoms with {func}`molsysmt.interactions.hbonds.get_acceptor_atoms`.
- {ref}`Get donor atoms <Tutorial_Get_donor_atoms>`: Identify donor atoms with {func}`molsysmt.interactions.hbonds.get_donor_atoms`.
:::
